In [49]:
import plotly.graph_objects as go

In [ ]:
VIEWPOINTS = {
    "front":        dict(x=0.0,  y=-2.0, z=0.0),
    "back":         dict(x=0.0,  y=2.0,  z=0.0),
    "left":         dict(x=-2.0, y=0.0,  z=0.0),
    "right":        dict(x=2.0,  y=0.0,  z=0.0),
    "top":          dict(x=0.0,  y=0.0,  z=2.0),
    "bottom":       dict(x=0.0,  y=0.0,  z=-2.0),
    "iso":          dict(x=1.45, y=-1.45, z=0.95),
    "aperture":     dict(x=-2.0, y=1.2,  z=0.65),
    "presentation": dict(x=-2.0, y=1.2,  z=1.45),
}

In [ ]:
def create_shell_trace(shell_mesh, options):
    """
    Create the main Plotly Mesh3d trace for the shell surface.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param options: Dictionary of plotting options
    :return: Plotly Mesh3d trace
    """
    X, Y, Z, C, I, J, K, _ = shell_mesh

    return go.Mesh3d(
        x=X,
        y=Y,
        z=Z,
        i=I,
        j=J,
        k=K,
        intensity=C,
        colorscale="YlOrBr",
        showscale=False,
        opacity=options["shell_opacity"],
        flatshading=False,
        lighting=dict(
            ambient=0.35,
            diffuse=0.95,
            specular=1.2,
            roughness=0.12,
            fresnel=0.5
        )
    )

# Construct a wireframe representation of the shell mesh

The shell surface is stored as a triangle mesh.

Each triangle is defined by three vertex indices:

    (i, j, k)

For example:

    i = 42
    j = 43
    k = 58

represents a triangle connecting vertices:

    42 ----- 43
      \     /
       \   /
         58

A wireframe view requires the set of unique edges in the mesh:

    (42,43)
    (43,58)
    (58,42)

Because neighbouring triangles share edges, many edges appear twice
in the triangle list.

Example:

    Triangle A -> (42,43,58)
    Triangle B -> (43,42,61)

Both triangles contain the same geometric edge:

    42 ----- 43

but one triangle references it as:

    (42,43)

and the other as:

    (43,42)

To ensure both representations are recognised as the same edge, the
vertex pair is sorted before storage:

    (43,42) -> (42,43)

A Python set is then used to automatically eliminate duplicates,
leaving only one copy of each unique edge in the mesh.

The resulting edge set forms the topological skeleton of the shell
and can be rendered as a wireframe overlay.

In [52]:
def extract_mesh_edges(I, J, K):
    """
    Extract unique edges from triangle index arrays.

    :param I: Triangle index array I
    :param J: Triangle index array J
    :param K: Triangle index array K
    :return: Set of unique vertex-index edge pairs
    """
    edges = set()

    for i, j, k in zip(I, J, K):
        edges.add(tuple(sorted((i, j))))
        edges.add(tuple(sorted((j, k))))
        edges.add(tuple(sorted((k, i))))

    return edges

In [53]:
def build_wireframe_coordinates(X, Y, Z, edges):
    """
    Convert mesh edges into Plotly line-coordinate arrays.

    :param X: Mesh x coordinates
    :param Y: Mesh y coordinates
    :param Z: Mesh z coordinates
    :param edges: Set of unique vertex-index edge pairs
    :return: edge_x, edge_y, edge_z coordinate lists
    """
    edge_x = []
    edge_y = []
    edge_z = []

    for a, b in edges:
        edge_x.extend([X[a], X[b], None])
        edge_y.extend([Y[a], Y[b], None])
        edge_z.extend([Z[a], Z[b], None])

    return edge_x, edge_y, edge_z

In [ ]:
def create_wireframe_trace(shell_mesh):
    """
    Create a Plotly Scatter3d wireframe trace from the shell mesh.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :return: Plotly Scatter3d trace
    """
    X, Y, Z, _, I, J, K, _ = shell_mesh

    edges = extract_mesh_edges(I, J, K)
    edge_x, edge_y, edge_z = build_wireframe_coordinates(X, Y, Z, edges)

    return go.Scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode="lines",
        line=dict(
            color="black",
            width=1
        ),
        showlegend=False
    )

In [ ]:
def create_chamber_trace(chamber_mesh, options):
    """
    Create a Plotly Mesh3d trace for internal chamber septa.

    :param chamber_mesh: Tuple returned by build_chamber_septa
    :param options: Dictionary of plotting options
    :return: Plotly Mesh3d trace
    """
    Xc, Yc, Zc, Cc, Ic, Jc, Kc, _ = chamber_mesh

    return go.Mesh3d(
        x=Xc,
        y=Yc,
        z=Zc,
        i=Ic,
        j=Jc,
        k=Kc,
        intensity=Cc,
        colorscale="YlOrBr",
        opacity=options["septa_opacity"],
        flatshading=False,
        showscale=False
    )

In [ ]:
def create_siphuncle_trace(siphuncle_mesh, options):
    """
    Create a Plotly Mesh3d trace for the siphuncle tube.

    :param siphuncle_mesh: Tuple returned by build_siphuncle_mesh
    :param options: Dictionary of plotting options
    :return: Plotly Mesh3d trace
    """
    Xs, Ys, Zs, Is, Js, Ks, _ = siphuncle_mesh

    return go.Mesh3d(
        x=Xs,
        y=Ys,
        z=Zs,
        i=Is,
        j=Js,
        k=Ks,
        color="rgb(180,120,80)",
        opacity=options["siphuncle_opacity"],
        flatshading=False,
        showscale=False
    )

In [57]:
def build_plot_traces(options, shell_mesh, chamber_mesh=None, siphuncle_mesh=None):
    """
    Build all Plotly traces required for the shell visualisation.

    :param options: Dictionary of plotting options
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    :return: List of Plotly traces
    """
    traces = [create_shell_trace(shell_mesh, options)]

    if options["show_wireframe"]:
        traces.append(create_wireframe_trace(shell_mesh))

    if chamber_mesh is not None:
        traces.append(create_chamber_trace(chamber_mesh, options))

    if siphuncle_mesh is not None:
        traces.append(create_siphuncle_trace(siphuncle_mesh, options))

    return traces

In [58]:
def apply_shell_layout(fig, options):
    """
    Apply title, sizing, background, axis, and aspect-ratio settings.

    :param fig: Plotly Figure object
    :param options: Dictionary of plotting options
    """
    title = dict(
        text=options["title"],
        x=0.5,
        xanchor="center"
    )

    if options["hide_axes"]:
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                bgcolor="rgba(0,0,0,0)",
                aspectmode="data"
            ),
            margin=dict(l=0, r=0, t=0, b=0),
            paper_bgcolor="rgba(0,0,0,1)",
            plot_bgcolor="rgba(0,0,0,1)",
            width=options["width"],
            height=options["height"]
        )
    else:
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title="x",
                yaxis_title="y",
                zaxis_title="z",
                aspectmode="data"
            ),
            width=options["width"],
            height=options["height"]
        )

In [ ]:
def plot_shell_mesh(
    options,
    shell_mesh,
    chamber_mesh=None,
    siphuncle_mesh=None,
    viewpoint="iso",
    zoom=1.0
):
    """
    Render the procedural shell geometry using Plotly Mesh3d.

    :param options: Dictionary of plotting options
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    :param viewpoint: Named viewpoint, or a Plotly camera eye dictionary
    :param zoom: Zoom factor (camera distance); smaller zooms in
    """
    traces = build_plot_traces(
        options,
        shell_mesh,
        chamber_mesh=chamber_mesh,
        siphuncle_mesh=siphuncle_mesh
    )

    fig = go.Figure(data=traces)

    apply_shell_layout(fig, options)

    if isinstance(viewpoint, str):
        if viewpoint not in VIEWPOINTS:
            raise ValueError(
                f"Unknown viewpoint '{viewpoint}'. "
                f"Available viewpoints: {list(VIEWPOINTS.keys())}"
            )
        camera_eye = VIEWPOINTS[viewpoint]
    else:
        camera_eye = viewpoint

    scaled_camera_eye = {k: v * zoom for k, v in camera_eye.items()}

    fig.update_layout(
        scene_camera=dict(
            eye=scaled_camera_eye
        )
    )

    fig.show()

In [ ]:
def plot_and_save_shell_mesh(
    options,
    shell_mesh,
    png_path,
    chamber_mesh=None,
    siphuncle_mesh=None,
    viewpoint="iso",
    zoom=1.0,
    width=1400,
    height=1000,
    scale=2,
    show=False,
):
    """
    Render the procedural shell geometry from a named viewpoint and save it
    as a PNG image.

    :param options: Dictionary of plotting options
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param png_path: Output PNG path
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    :param viewpoint: Named viewpoint, or a Plotly camera eye dictionary
    :param zoom: Zoom factor (camera distance); smaller zooms in
    :param width: Output image width in pixels
    :param height: Output image height in pixels
    :param scale: Image export scale factor
    :param show: Whether to display the figure interactively
    """

    traces = build_plot_traces(
        options,
        shell_mesh,
        chamber_mesh=chamber_mesh,
        siphuncle_mesh=siphuncle_mesh,
    )

    fig = go.Figure(data=traces)

    apply_shell_layout(fig, options)

    if isinstance(viewpoint, str):
        if viewpoint not in VIEWPOINTS:
            raise ValueError(
                f"Unknown viewpoint '{viewpoint}'. "
                f"Available viewpoints: {list(VIEWPOINTS.keys())}"
            )
        camera_eye = VIEWPOINTS[viewpoint]
    else:
        camera_eye = viewpoint

    scaled_camera_eye = {k: v * zoom for k, v in camera_eye.items()}

    fig.update_layout(
        scene_camera=dict(
            eye=scaled_camera_eye
        )
    )

    fig.write_image(
        png_path,
        width=width,
        height=height,
        scale=scale,
    )

    if show:
        fig.show()

    return fig